# Ordinary Least-Squares Regression for Randomized Control Trials Example

In [1]:
from cais.agent import CausalAgent
from IPython.display import Markdown
import pandas as pd

info_dir = "../../CauSciBench/data/real_info.csv"
data_dir = "../../CauSciBench/data/real_data/"

info = pd.read_csv(info_dir, encoding='latin-1')
selected = info.iloc[6]

data_file = selected['data_files']
description = selected['description']
ground_truth = selected['answer']
query = selected['natural_language_query']

print(f"{query = }")
print(f"{data_file = }")
print(f"description = ")
Markdown(description)

/Users/tae/anaconda3/envs/cais/lib/python3.10/site-packages/google/api_core/_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/tae/anaconda3/envs/cais/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


query = 'Does being an immigrant make it less likely to get an interview request?'
data_file = 'vernby_2019.csv'
description = 


The data comes from a randomized field experiment designed to assess whether a candidate's background influences their likelihood of receiving a job interview or offer. Researchers submitted fictitious job applications to restaurants and cafes across Sweden at random. The applications varied in terms of country of birth, gender, citizenship status, work experience, and religious activity. A positive response was defined as a job offer, interview invitation, or follow-up inquiry, while a negative response included any other reply or no response at all. Variables: name: Name of the candidate; stad: City; citizen: 1 if the candidate is a Swedish citizen, 0 otherwise; religious: 1 if the candidate is religious, 0 otherwise; experience: 1 if the candidate has work experience, 0 otherwise; poland: 1 if the candidate was born in Poland, 0 otherwise; iraq: 1 if the candidate was born in Iraq, 0 otherwise; somalia: 1 if the candidate was born in Somalia, 0 otherwise; skilledjob: 1 if the job is high-skilled, 0 otherwise; woman: 1 if the candidate is a woman, 0 otherwise; invited: 1 if the candidate received an interview or a job or a follow-up response, 0 otherwise; city1, city2, city3, city4, city5, city6, city7: Dummy variables for the seven cities; immigrant: 1 if the candidate is an immigrant (not born in Sweden), 0 otherwise; time: Proportion of the applicant's life spent living in Sweden (scaled between 0 and 1)

In [2]:
agent = CausalAgent(
    dataset_path = data_dir + data_file,
    dataset_description = description,
    model_name = 'gpt-4o-mini',
    provider = 'litellm'
) # construct agent

In [3]:
agent.analyse_dataset(
    query = query
)

13:57:09 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
13:57:11 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
13:57:11 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
13:57:13 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
13:57:13 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
13:57:17 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler


Interpreting query with hybrid approach...
Identified Confounders: ['citizen', 'religious', 'experience', 'skilledjob', 'woman', 'time']


In [4]:
agent.select_method(
    query = query,
    llm_decision=False
)

'regression_discontinuity_design'

In [5]:
agent.select_method(
    query = query,
    llm_decision=True
)

'linear_regression'

In [6]:
agent.select_controls() # can pass a query or use the most recently used one

Using LLM to refine covariate list for controls selection
Method Name:  linear_regression
LLM parsed result: covariates=['citizen', 'religious', 'experience', 'skilledjob', 'woman', 'city1', 'city2', 'city3', 'city4', 'city5', 'city6', 'city7'] reasoning=None
LLM refined controls to: ['citizen', 'religious', 'experience', 'skilledjob', 'woman', 'city1', 'city2', 'city3', 'city4', 'city5', 'city6', 'city7']


In [7]:
agent.clean_dataset() # generates code to clean the dataset, subject to strict guidelines

'/Users/tae/Work/zhijing/CauSciBench/data/real_data/vernby_2019_cleaned_25604.csv'

In [8]:
agent.execute_method()

effect_estimates_by_level: {'treatment_effect': {'estimate': np.float64(-0.08611704628932033), 'p_value': np.float64(0.0005900321757261033), 'conf_int': [-0.13517178151694892, -0.03706231106169176], 'std_err': np.float64(0.025007972932280253)}}
LLM interpretation raw output: The analysis indicates that being an immigrant is associated with a decrease in the likelihood of receiving an interview request, with an estimated effect of -0.086 (SE = 0.025), and a 95% confidence interval of [-0.135, -0.037]. This effect is statistically significant (p < 0.001), suggesting that immigrants are indeed less likely to be invited for interviews compared to non-immigrants. The use of linear regression is plausible given the randomized controlled trial design, which minimizes bias and allows for the control of confounding variables. This method is appropriate as it effectively estimates the treatment effect while accounting for relevant covariates, whereas alternative methods may not adequately addres

{'results': {'effect_estimate': np.float64(-0.08611704628932033),
  'p_value': np.float64(0.0005900321757261033),
  'confidence_interval': [-0.13517178151694892, -0.03706231106169176],
  'standard_error': np.float64(0.025007972932280253),
  'estimated_effects_by_level': None,
  'reference_level_used': None,
  'formula': 'invited ~ immigrant + citizen + religious + experience + skilledjob + woman + time',
  'model_summary_text': '                            OLS Regression Results                            \n==============================================================================\nDep. Variable:                invited   R-squared:                       0.069\nModel:                            OLS   Adj. R-squared:                  0.065\nMethod:                 Least Squares   F-statistic:                     15.83\nDate:                Sat, 18 Apr 2026   Prob (F-statistic):           3.72e-20\nTime:                        13:59:29   Log-Likelihood:                -396.79\nNo. Obs

In [9]:
agent.results['effect_estimate']

In [10]:
ground_truth

In [11]:
Markdown(agent.explanations['final_explanation_text'])

**Method Used:** Linear Regression

**Method Explanation:**
The linear_regression method is a causal inference technique used to estimate causal effects from observational data.

**Results:**
- Estimated Causal Effect: -0.0861
- 95% Confidence Interval: [-0.1352, -0.0371]
- P-value: 0.0006

**Interpretation Guide:**
The estimated effect represents the causal impact of immigrant on invited, given the assumptions of the method are met. Careful consideration of these assumptions is needed for valid causal interpretation.

**Assumptions:**
- linear relationship between treatment, covariates, and outcome: This is a key assumption for the selected causal inference method.
- no unmeasured confounders (if observational): This is a key assumption for the selected causal inference method.
- correct model specification: This is a key assumption for the selected causal inference method.
- homoscedasticity of errors: This is a key assumption for the selected causal inference method.
- normally distributed errors (for inference): This is a key assumption for the selected causal inference method.

**Limitations:**
The linear_regression method has general limitations in terms of its assumptions and applicability.



In [12]:
Markdown(agent.explanations['interpretation_text'])

The analysis indicates that being an immigrant is associated with a decrease in the likelihood of receiving an interview request, with an estimated effect of -0.086 (SE = 0.025), and a 95% confidence interval of [-0.135, -0.037]. This effect is statistically significant (p < 0.001), suggesting that immigrants are indeed less likely to be invited for interviews compared to non-immigrants. The use of linear regression is plausible given the randomized controlled trial design, which minimizes bias and allows for the control of confounding variables. This method is appropriate as it effectively estimates the treatment effect while accounting for relevant covariates, whereas alternative methods may not adequately address potential confounding. However, key threats to identification validity include the assumption of a linear relationship and the potential for unmeasured confounders, which could affect the robustness of the results. Limitations include the specific context of the study, which may not generalize to all job markets or immigrant populations.

In [13]:
#agent.run_analysis(query=query)